## 02 — Data Quality Audit

This notebook audits the raw Statcast + enriched frame before any analysis or modelling. Columns are organized into five EDA tiers (311 `EDA_COLUMNS` total): candidate game state, pitcher/batter usage, and pitcher/batter outcome blocks.

**Sections:**
1. Row-Level Sanity Checks
2. Missingness Analysis
3. Feature Type Classification

In [ ]:
%matplotlib inline

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import pybaseball
from pybaseball import cache, statcast

sys.path.append(str(Path("..").resolve()))
from utils.features.enrichment import enrich_statcast
from utils.features.feature_names import (
    BATTER_OUTCOME_COLUMNS,
    BATTER_USAGE_COLUMNS,
    CANDIDATE_GAME_STATE_COLUMNS,
    EDA_COLUMNS,
    LABEL_COLUMN,
    PITCHER_OUTCOME_COLUMNS,
    PITCHER_USAGE_COLUMNS,
    PITCH_TYPES,
)

DATA_DIR = Path("..") / "data" / "cache"
DATA_DIR.mkdir(parents=True, exist_ok=True)
cache.config.cache_directory = str(DATA_DIR.resolve())
cache.config.save()
cache.enable()

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

print("pybaseball :", pybaseball.__version__)
print("pandas     :", pd.__version__)
print("cache dir  :", cache.config.cache_directory)

In [ ]:
SEASON = 2024
PRIOR  = SEASON - 1

df = statcast("2024-06-01", "2024-06-30")
print(f"Raw shape: {df.shape}")

df_enr = enrich_statcast(df, PRIOR)
print(f"Enriched shape: {df_enr.shape}")
if "pitch_usage_FF" in df_enr.columns:
    print(f"Pitcher usage coverage (pitch_usage_FF non-null): {df_enr['pitch_usage_FF'].notna().mean():.1%}")

In [ ]:
available    = [c for c in EDA_COLUMNS if c in df_enr.columns]
missing_cols = [c for c in EDA_COLUMNS if c not in df_enr.columns]

model_df = df_enr[available + [LABEL_COLUMN]].copy()
model_df = model_df[model_df[LABEL_COLUMN].isin(PITCH_TYPES)].reset_index(drop=True)

print(f"model_df shape       : {model_df.shape}")
print(f"EDA_COLUMNS total    : {len(EDA_COLUMNS)}")
print(f"Found in enriched frame: {len(available)}")
if missing_cols:
    preview = missing_cols[:5]
    ellipsis = "..." if len(missing_cols) > 5 else ""
    print(f"Missing ({len(missing_cols)})           : {preview}{ellipsis}")
print(f"\nPitch type distribution:\n{model_df[LABEL_COLUMN].value_counts()}")

---
## Section 1 — Row-Level Sanity Checks

Confirm the raw pull covers the expected date window, contains no accidental duplicates, and that filtering to known `PITCH_TYPES` doesn't silently discard a significant fraction of pitches.

In [ ]:
### Date range, total rows, and pitch label coverage
print('Date range:', df['game_date'].min(), '→', df['game_date'].max())
print('Raw rows  :', len(df))
print('Enriched  :', len(df_enr))
print('model_df  :', len(model_df), '(after filtering to known PITCH_TYPES)\n')

### All pitch_type values in the raw pull, before filtering
raw_counts = df['pitch_type'].value_counts(dropna=False)
in_types   = raw_counts[raw_counts.index.isin(PITCH_TYPES)]
out_types  = raw_counts[~raw_counts.index.isin(PITCH_TYPES)]
print(f'Pitch types kept ({in_types.sum():,} rows):')
print(in_types.to_string())
print(f'\nPitch types dropped ({out_types.sum():,} rows):')
print(out_types.to_string())

In [ ]:
### Duplicate row detection
KEY_COLS = ['game_pk', 'at_bat_number', 'pitch_number']

full_dups = df_enr.duplicated().sum()
key_dups  = df_enr.duplicated(subset=KEY_COLS).sum()

print(f'Full-row duplicates : {full_dups}')
print(f'Key-column duplicates ({KEY_COLS}): {key_dups}')
if key_dups > 0:
    print('\nSample duplicate keys:')
    dup_mask = df_enr.duplicated(subset=KEY_COLS, keep=False)
    print(df_enr[dup_mask][KEY_COLS + ['pitch_type']].head(10))

In [ ]:
### Pitches per game-date — flag sparse dates

per_date = df_enr.groupby('game_date').size().sort_index()

print(f'Total game dates : {len(per_date)}')
print(f'Median pitches/date: {per_date.median():.0f}')
fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(per_date.index.astype(str), per_date.values, width=0.8, color='steelblue')
ax.set_xlabel('Game date')
ax.set_ylabel('Pitch count')
ax.set_title('Pitches per Game Date — June 2024', fontsize=13)
ax.legend()
plt.xticks(rotation=45, ha='right', fontsize=7)
sns.despine()
plt.tight_layout()
plt.show()

---
## Section 2 — Missingness Analysis

Audits **EDA_COLUMNS** (311 cols) in five tiers: candidate game state, pitcher usage, pitcher outcome, batter usage, batter outcome. Also summarizes null rates across the full enriched frame.

High null rates in enrichment columns are expected for players missing from prior-year arsenal tables. Rare pitch types (KN, SV) often have structural gaps in outcome columns.

In [ ]:
### Null % for every column in the enriched frame, sorted descending
all_null = df_enr.isnull().mean().sort_values(ascending=False)

print(f'Columns with any null   : {(all_null > 0).sum()} / {len(all_null)}')
print(f'Columns fully null       : {(all_null == 1).sum()}')
print(f'Columns > 50% null       : {(all_null > 0.5).sum()}')
print(f'Columns > 20% null       : {(all_null > 0.2).sum()}')

In [ ]:
### Top 10 most-missing columns

top10_null = all_null[all_null > 0].head(10)
top10_df = top10_null.rename_axis('Column').reset_index()
top10_df.columns = ['Column', 'Missing %']
top10_df['Missing %'] = top10_df['Missing %'].map(lambda x: f'{x:.1%}')
top10_df.index = top10_df.index + 1
print(top10_df.to_string())

In [ ]:
### Missingness by EDA column tier
tier_map = {
    "Candidate game state": CANDIDATE_GAME_STATE_COLUMNS,
    "Pitcher usage": PITCHER_USAGE_COLUMNS,
    "Pitcher outcome": PITCHER_OUTCOME_COLUMNS,
    "Batter usage": BATTER_USAGE_COLUMNS,
    "Batter outcome": BATTER_OUTCOME_COLUMNS,
}

rows = []
for tier, cols in tier_map.items():
    present = [c for c in cols if c in model_df.columns]
    if not present:
        continue
    null_pct = model_df[present].isna().mean()
    rows.append({
        "tier": tier,
        "n_cols": len(present),
        "median_null_pct": null_pct.median(),
        "max_null_pct": null_pct.max(),
        "fully_null_cols": int((null_pct == 1.0).sum()),
    })

tier_summary = pd.DataFrame(rows).sort_values("median_null_pct", ascending=False)
display(tier_summary)

### Top 20 columns by null rate (all EDA columns)
null_all = model_df[available].isna().mean().sort_values(ascending=False)
null_all.head(20).to_frame("null_fraction")

---
## Section 3 — Feature Type Classification

Automated scan across every column in `model_df`. A column is classified as:
- **categorical** — object dtype, or integer/float with ≤ 15 unique non-null values
- **binary** — exactly 2 unique non-null values
- **continuous** — everything else

Columns flagged at the end have numeric dtypes but suspiciously low cardinality, which may indicate label-encoded categoricals that should be treated as such in modelling.

In [ ]:
CATEGORICAL_THRESHOLD = 15  # unique non-null values at or below → treat as categorical

classification = {}
for col in model_df.columns:
    s = model_df[col].dropna()
    n_unique = s.nunique()
    if n_unique == 2:
        classification[col] = 'binary'
    elif model_df[col].dtype == object or n_unique <= CATEGORICAL_THRESHOLD:
        classification[col] = 'categorical'
    else:
        classification[col] = 'continuous'

type_series = pd.Series(classification)
counts = type_series.value_counts()
print('Feature type counts:')
print(counts.to_string())

In [ ]:
### Categorical columns — unique values and cardinality
cat_cols = type_series[type_series == 'categorical'].index.tolist()

cat_rows = []
for col in cat_cols:
    uniq = model_df[col].dropna().unique()
    cat_rows.append({
        'Column':      col,
        'Cardinality': len(uniq),
        'Values':      ', '.join(sorted(str(v) for v in uniq)[:10]) + ('...' if len(uniq) > 10 else ''),
    })

cat_df = pd.DataFrame(cat_rows).sort_values('Cardinality', ascending=False)
print(cat_df.to_string(index=False))